In [ ]:
from myosuite.utils import gym
import imageio
from IPython.display import Video, display
import numpy as np
import os
import pickle


MyoSuite:> Registering Myo Envs


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:

pth = '../myosuite/agents/baslines_NPG/'
policy = os.path.join(
    pth,
    "myoElbowPose1D6MExoRandom-v0/2022-02-26_21-16-27/36_env=myoElbowPose1D6MExoRandom-v0,seed=1/iterations/best_policy.pickle",
)

if not os.path.exists(policy):
    print(f"Policy file not found at {policy}, skipping policy loading in this environment.")
    pi = None
else:
    with open(policy, 'rb') as f:
        pi = pickle.load(f)


In [3]:
env = gym.make('myoElbowPose1D6MExoRandom-v0')

env.reset()


    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    


c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


(array([0.0332, 0.    , 0.43  , 0.    , 0.    , 0.    , 0.    , 0.    ,
        0.    ], dtype=float32),
 {})

In [7]:
# define a discrete sequence of positions to test
AngleSequence = [60, 30, 30, 60, 80, 80, 60, 30, 80, 30, 80, 60]

frames = []

if pi is None:
    print("Skipping rollout and video generation because the policy file is unavailable.")
else:
    env.reset()
    for ep in range(len(AngleSequence)):
        print("Ep {} of {} testing angle {}".format(ep, len(AngleSequence), AngleSequence[ep]))
        env.unwrapped.target_jnt_value = [np.deg2rad(AngleSequence[int(ep)])]
        env.unwrapped.target_type = 'fixed'
        env.unwrapped.weight_range = (0, 0)
        env.unwrapped.update_target()
        for _ in range(40):
            frame = env.unwrapped.mj_renderer.render_offscreen(
                width=400,
                height=400,
                camera_id=0,
            )
            frames.append(frame)
            o = env.unwrapped.get_obs()
            a = pi.get_action(o)[0]
            next_o, r, done, *_, ifo = env.step(a)  # take an action based on the current observation
    env.close()

if frames:
    os.makedirs('videos', exist_ok=True)
    # make a local copy
    imageio.mimwrite(f"videos/exo_arm.mp4", frames, fps=1.0 / env.unwrapped.dt)


Ep 0 of 12 testing angle 60
Ep 1 of 12 testing angle 30
Ep 2 of 12 testing angle 30
Ep 3 of 12 testing angle 60
Ep 4 of 12 testing angle 80
Ep 5 of 12 testing angle 80
Ep 6 of 12 testing angle 60
Ep 7 of 12 testing angle 30
Ep 8 of 12 testing angle 80
Ep 9 of 12 testing angle 30
Ep 10 of 12 testing angle 80
Ep 11 of 12 testing angle 60


In [6]:
display(Video(f"videos/exo_arm.mp4", embed=True))